## SI-C for MFA data preparation, case study O'ahu

All pre-data handling before inserting into SI-C_calculations is described here. 

In [1]:
import pandas as pd 
import numpy as np

### Imports and Exports

#### 1) Available Data for Oahu  
Different FAF frameworks are used:  (insert overview table)  

There are three different types of trade (called trade type in the data base): 
| No. | Description                                                                                   |
|-----|-----------------------------------------------------------------------------------------------|
|  1  | Domestic flows (freight shipments moved from US domestic origins to US domestic destinations) |
|  2  | Import flows (freight shipments moved from foreign origins to US domestic destinations)       |
|  3  | Export flows (freight shipments moved from US domestic origins to foreign destinations)       |

Because of the definitions given in the table: 
* domestic flows could be imports or exports to / from Hawaii.
* import flows could be imports or exports to / from Hawaii.
* export flows could be exports or imports from / to Hawaii.  
→ it is necessary to look at the fr_orig, dms_origst, dms_destst and fr_dest information 

There are different types of foreign FAF zones:
| Numeric Label | Description       |
|---------------|-------------------|
|      801      | Canada            |
|      802      | Mexico            |
|      803      | Rest of Americas  |
|      804      | Europe            |
|      805      | Africa            |
|      806      | SW & Central Asia |
|      807      | Eastern Asia      |
|      808      | SE Asia & Oceania |

And for Hawaii / Oahu flows, only two domestic zones are relevant:
* 151: Honolulu HI
* 159: Rest of HI

##### 1.0) General Method
Different kind of flows are defined to find out what is being exported or imported from / to Oahu / Hawaii.  

__COUNTY DATA: OAHU__
* NaN > Oahu > US OR Rest of HI > NaN
  * This flow is considered as an __Export__ from Oahu. Source = Oahu, Target = US mainland or Rest of HI (Rest of Hawaii)
* NaN > Oahu > US OR Rest of HI > RoW
  * This flow is considered as an __Export__ from Oahu. Source = Oahu, Target = US mainland or Rest of HI
* NaN > Oahu > Oahu > RoW
  * This flow is considered as an __Export__ from Oahu. Source = Oahu, Target = RoW
    
* RoW > US OR Rest of HI > Oahu > NaN
  * This flow is considered as an __Import__ to Oahu. Source = US mainland or Rest of HI, Target = Oahu
* NaN > US OR Rest of HI > Oahu > NaN
  * This flow is considered as an __Import__ to Oahu. Source = US mainland or Rest of HI, Target = Oahu
* RoW > Oahu > Oahu > NaN
  * This flow is considered as an __Import__ to Oahu. Source = RoW, Target = Oahu
 

* (NaN > Oahu > Rest of HI > RoW)
  * (This flow is considered as an __Export__ from Oahu. Source = Oahu, Target = Rest of HI)
* (RoW > US > Oahu > NaN)
  * (This flow is considered as an __Import__ to Oahu. Source = US mainland, Target = Oahu)
 
__STATE DATA: HAWAII__  
* NaN > Hawaii > US > NaN
  * This flow is considered as an __Export__ from Hawaii. Source = Hawaii, Target = US mainland
* NaN > Hawaii > US > RoW
  * This flow is considered as an __Export__ from Hawaii. Source = Hawaii, Target = US mainland
* NaN > Hawaii > Hawaii > RoW
  * This flow is considered as an __Export__ from Hawaii. Source = Hawaii, Target = RoW (Rest of World)
* RoW > Hawaii > Hawaii > NaN
  * This flow is considered as an __Import__ to Hawaii. Source = RoW, Target = Hawaii
* NaN > US > Hawaii > NaN
  * This flow is considered as an __Import__ to Hawaii. Source = US mainland, Target = Hawaii
* RoW > US > Hawaii > NaN
  * This flow is considered as an __Import__ to Hawaii. Source = US mainland, Target = Hawaii  

REVISION: We further need to include domestic flows within Hawaii. This allows us to calculate shares for international and domestic trade flows.
* NAN > Hawaii > Hawaii > NaN
  * These can be both imports and exports, depending on the perspective.

##### 1.1) 2013-2017  
For this time period, the regional (county-level) FAF4.5.1 is used.

In [3]:
years = [2013, 2014, 2015, 2016, 2017]

In [5]:
#sctg2_mapping = "ODYM-master/new_model/FAF_mapping.xlsx" path to sctg2 mapping
FAF4_sctg2 = pd.read_excel("C:/Users/linsmasa/ODYM-master/revised_model/SI2_revised.xlsx", sheet_name = "FAF4")
# Dictionary for description 
desc_map = FAF4_sctg2.set_index("Code")["Commodity Description"]
mat_map = FAF4_sctg2.set_index("Code")["Material"]

Download link to FAF4.5.1: https://www.bts.gov/faf

In [8]:
def process_MFA(year):
    file_path = f"C:/Users/linsmasa/ODYM-master/new_model/Imports_Exports/FAF/FAF4.5.1_{year}.csv"
    df = pd.read_csv(file_path)

    # Remove columns 11-14 (0-based index, so it's columns 10-13)
    df = df.drop(df.columns[10:14], axis=1)

    # Filter for rows where dms_orig or dms_dest is 151 (Honolulu)
    origin151 = df[df['dms_orig'] == 151]
    dest151 = df[(df['dms_dest'] == 151) & (df['dms_orig'] != 151)]

    # Combine the filtered datasets
    Oahu = pd.concat([origin151, dest151], ignore_index=True)

    # Python reads some columns not as numbers
    Oahu['dms_orig'] = pd.to_numeric(Oahu['dms_orig'], errors='coerce')
    Oahu['dms_dest'] = pd.to_numeric(Oahu['dms_dest'], errors='coerce')
    Oahu['fr_dest'] = Oahu['fr_dest'].fillna(-1).astype(int)  # Replace NaN with -1, then convert to int
    Oahu['fr_orig'] = Oahu['fr_orig'].fillna(-1).astype(int)

    # Mapping function for locations
    def map_location(value):
        if value == -1:  # Skip processing if value is -1
            return -1
        if value == 151:
            return "Honolulu"
        elif value == 159:
            return "Imports from Rest of HI"
        elif 801 <= value <= 808:
            return {
                801: "Imports from Canada",
                802: "Imports from Mexico",
                803: "Imports from Rest of Americas",
                804: "Imports from Europe",
                805: "Imports from Africa",
                806: "Imports from SW & Central Asia",
                807: "Imports from Eastern Asia",
                808: "Imports from SE Asia & Oceania"
            }.get(value, value)
        else:
            return "Imports from US mainland"
    
    Oahu['dms_orig'] = Oahu['dms_orig'].apply(map_location)
    Oahu['fr_orig'] = Oahu['fr_orig'].apply(map_location)
    
    def map_location(value):
        if value == -1:  # Skip processing if value is -1
            return -1
        if value == 151:
            return "Honolulu"
        elif value == 159:
            return "Exports to Rest of HI"
        elif 801 <= value <= 808:
            return {
                801: "Exports to Canada",
                802: "Exports to Mexico",
                803: "Exports to Rest of Americas",
                804: "Exports to Europe",
                805: "Exports to Africa",
                806: "Exports to SW & Central Asia",
                807: "Exports to Eastern Asia",
                808: "Exports to SE Asia & Oceania"
            }.get(value, value)
        else:
            return "Exports to US mainland"
    
    Oahu['dms_dest'] = Oahu['dms_dest'].apply(map_location)
    Oahu['fr_dest'] = Oahu['fr_dest'].apply(map_location)

    # Define filtering options
    part1 = Oahu[(Oahu['fr_orig'] == -1) & (Oahu['dms_orig'] == "Honolulu")  & (Oahu['dms_dest'] != "Honolulu") & (Oahu['fr_dest'] == -1)].drop(columns=['fr_orig', 'fr_dest']).rename(columns={'dms_orig': 'Source', 'dms_dest': 'Target'})
    part2 = Oahu[(Oahu['fr_orig'] == -1) & (Oahu['dms_orig'] == "Honolulu")  & (Oahu['dms_dest'] != "Honolulu") & (Oahu['fr_dest'] != -1)].drop(columns=['fr_orig', 'fr_dest']).rename(columns={'dms_orig': 'Source', 'dms_dest': 'Target'})
    part3 = Oahu[(Oahu['fr_orig'] == -1) & (Oahu['dms_orig'] == "Honolulu")  & (Oahu['dms_dest'] == "Honolulu") & (Oahu['fr_dest'] != -1)].drop(columns=['fr_orig', 'dms_orig']).rename(columns={'dms_dest': 'Source', 'fr_dest': 'Target'})
    part4 = Oahu[(Oahu['fr_orig'] != -1) & (Oahu['dms_orig'] != "Honolulu")  & (Oahu['dms_dest'] == "Honolulu") & (Oahu['fr_dest'] == -1)].drop(columns=['fr_orig', 'fr_dest']).rename(columns={'dms_orig': 'Source', 'dms_dest': 'Target'})
    part5 = Oahu[(Oahu['fr_orig'] == -1) & (Oahu['dms_orig'] != "Honolulu")  & (Oahu['dms_dest'] == "Honolulu") & (Oahu['fr_dest'] == -1)].drop(columns=['fr_orig', 'fr_dest']).rename(columns={'dms_orig': 'Source', 'dms_dest': 'Target'})
    part6 = Oahu[(Oahu['fr_orig'] != -1) & (Oahu['dms_orig'] == "Honolulu")  & (Oahu['dms_dest'] == "Honolulu") & (Oahu['fr_dest'] == -1)].drop(columns=['dms_dest', 'fr_dest']).rename(columns={'fr_orig': 'Source', 'dms_orig': 'Target'})
    
    # Combine all parts
    df_final = pd.concat([part1, part2, part3, part4, part5, part6], ignore_index=True)

    # Rename columns
    df_final.columns = ["Source", "Target", "fr_inmode", "dms_mode", "fr_outmode", "sctg2", "trade_type", "Tons"]

    # Map SCTG product codes to product names
    df_final["Commodity Description"] = df_final["sctg2"].map(desc_map)
    df_final["Material"] = df_final["sctg2"].map(mat_map)

    df_final = df_final.rename(columns={"Source": "Origin", "Target": "Destination"})
    df_final['Source'] = np.where(df_final["Origin"].str.contains("Imports", case=False, na=False), "Imports", "NAS")
    df_final['Target'] = np.where(df_final["Destination"].str.contains("Exports", case=False, na=False), "Exports", "DMI")
    
    # add year column for comparison later
    df_final['Year'] = year

    # Save results
    output_file = f"C:/Users/linsmasa/ODYM-master/revised_model/Imports_Exports/Oahu_MFA{year}.xlsx"
    df_final.to_excel(output_file, index=False)
    print(f"Processed {year} and saved to {output_file}")

In [10]:
for year in years:
    process_MFA(year)

Processed 2013 and saved to C:/Users/linsmasa/ODYM-master/revised_model/Imports_Exports/Oahu_MFA2013.xlsx
Processed 2014 and saved to C:/Users/linsmasa/ODYM-master/revised_model/Imports_Exports/Oahu_MFA2014.xlsx
Processed 2015 and saved to C:/Users/linsmasa/ODYM-master/revised_model/Imports_Exports/Oahu_MFA2015.xlsx
Processed 2016 and saved to C:/Users/linsmasa/ODYM-master/revised_model/Imports_Exports/Oahu_MFA2016.xlsx
Processed 2017 and saved to C:/Users/linsmasa/ODYM-master/revised_model/Imports_Exports/Oahu_MFA2017.xlsx


##### 1.2) 2018-2022  
For this time period, the regional (county-level) FAF5.6.1 is used.

In [13]:
years = [2018, 2019, 2020, 2021, 2022, 2023]

In [15]:
FAF5_sctg2 = pd.read_excel("C:/Users/linsmasa/ODYM-master/revised_model/SI2_revised.xlsx", sheet_name = "FAF5")
desc_map = FAF5_sctg2.set_index("Code")["Commodity Description"]
mat_map = FAF5_sctg2.set_index("Code")["Material"]

Download link to FAF5.6.1: https://www.bts.gov/faf

In [18]:
Oahu = pd.read_csv("C:/Users/linsmasa/ODYM-master/revised_model/Imports_Exports/FAF5.6.1_2018-2023.csv")
# Keep only the first 15 columns (removing columns 16-30) (reduce the size of df)
Oahu = Oahu.iloc[:, :16]
df = Oahu
# Filter for rows where dms_orig or dms_dest is 151 (Honolulu)
origin151 = df[df['dms_orig'] == 151]
dest151 = df[df['dms_dest'] == 151]

# Remove rows where dms_orig == dms_dest (both 151), is only deleted in one of them, so there are no duplicates
dest151 = dest151[dest151['dms_orig'] != 151]

# Combine filtered data
Oahu = pd.concat([origin151, dest151], ignore_index=True)
# Python reads some columns not as numbers
Oahu['dms_orig'] = pd.to_numeric(Oahu['dms_orig'], errors='coerce')
Oahu['dms_dest'] = pd.to_numeric(Oahu['dms_dest'], errors='coerce')
Oahu['fr_dest'] = Oahu['fr_dest'].fillna(-1).astype(int)  # Replace NaN with -1, then convert to int
Oahu['fr_orig'] = Oahu['fr_orig'].fillna(-1).astype(int)

# Replace numeric locations with names, first dms_orig
def map_location(value):
    if value == -1:  # Skip processing if value is -1
        return -1
    if value == 151:
        return "Honolulu"
    elif value == 159:
        return "Imports from Rest of HI"
    elif 801 <= value <= 808:
        return {
            801: "Imports from Canada",
            802: "Imports from Mexico",
            803: "Imports from Rest of Americas",
            804: "Imports from Europe",
            805: "Imports from Africa",
            806: "Imports from SW & Central Asia",
            807: "Imports from Eastern Asia",
            808: "Imports from SE Asia & Oceania"
        }.get(value, value)
    else:
        return "Imports from US mainland"

Oahu['dms_orig'] = Oahu['dms_orig'].apply(map_location)
Oahu['fr_orig'] = Oahu['fr_orig'].apply(map_location)

def map_location(value):
    if value == -1:  # Skip processing if value is -1
        return -1
    if value == 151:
        return "Honolulu"
    elif value == 159:
        return "Exports to Rest of HI"
    elif 801 <= value <= 808:
        return {
            801: "Exports to Canada",
            802: "Exports to Mexico",
            803: "Exports to Rest of Americas",
            804: "Exports to Europe",
            805: "Exports to Africa",
            806: "Exports to SW & Central Asia",
            807: "Exports to Eastern Asia",
            808: "Exports to SE Asia & Oceania"
        }.get(value, value)
    else:
        return "Exports to US mainland"

Oahu['dms_dest'] = Oahu['dms_dest'].apply(map_location)
Oahu['fr_dest'] = Oahu['fr_dest'].apply(map_location)

part1 = Oahu[(Oahu['fr_orig'] == -1) & (Oahu['dms_orig'] == "Honolulu")  & (Oahu['dms_dest'] != "Honolulu") & (Oahu['fr_dest'] == -1)].drop(columns=['fr_orig', 'fr_dest']).rename(columns={'dms_orig': 'Source', 'dms_dest': 'Target'})
part2 = Oahu[(Oahu['fr_orig'] == -1) & (Oahu['dms_orig'] == "Honolulu")  & (Oahu['dms_dest'] != "Honolulu") & (Oahu['fr_dest'] != -1)].drop(columns=['fr_orig', 'fr_dest']).rename(columns={'dms_orig': 'Source', 'dms_dest': 'Target'})
part3 = Oahu[(Oahu['fr_orig'] == -1) & (Oahu['dms_orig'] == "Honolulu")  & (Oahu['dms_dest'] == "Honolulu") & (Oahu['fr_dest'] != -1)].drop(columns=['fr_orig', 'dms_orig']).rename(columns={'dms_dest': 'Source', 'fr_dest': 'Target'})
part4 = Oahu[(Oahu['fr_orig'] != -1) & (Oahu['dms_orig'] != "Honolulu")  & (Oahu['dms_dest'] == "Honolulu") & (Oahu['fr_dest'] == -1)].drop(columns=['fr_orig', 'fr_dest']).rename(columns={'dms_orig': 'Source', 'dms_dest': 'Target'})
part5 = Oahu[(Oahu['fr_orig'] == -1) & (Oahu['dms_orig'] != "Honolulu")  & (Oahu['dms_dest'] == "Honolulu") & (Oahu['fr_dest'] == -1)].drop(columns=['fr_orig', 'fr_dest']).rename(columns={'dms_orig': 'Source', 'dms_dest': 'Target'})
part6 = Oahu[(Oahu['fr_orig'] != -1) & (Oahu['dms_orig'] == "Honolulu")  & (Oahu['dms_dest'] == "Honolulu") & (Oahu['fr_dest'] == -1)].drop(columns=['dms_dest', 'fr_dest']).rename(columns={'fr_orig': 'Source', 'dms_orig': 'Target'})

# Combine all parts
df_final = pd.concat([part1, part2, part3, part4, part5, part6], ignore_index=True)
df_final.head()

# Map SCTG product codes to product names
df_final["Commodity Description"] = df_final["sctg2"].map(desc_map)
df_final["Material"] = df_final["sctg2"].map(mat_map)

In [20]:
# change the format from wide to long
# Automatically detect year columns (those starting with 'tons_')
year_cols = [c for c in df_final.columns if c.startswith("tons_")]

# Melt while keeping all other columns
df_long = df_final.melt(
    id_vars=[c for c in df_final.columns if c not in year_cols],
    value_vars=year_cols,
    var_name='Year',
    value_name='Tons'
)

# Extract numeric year from 'kg_2018' → 2018
df_long['Year'] = df_long['Year'].str.extract(r'(\d{4})').astype(int)

df_final = df_long.copy()
df_final = df_final.rename(columns={"Source": "Origin", "Target": "Destination"})
df_final['Source'] = np.where(df_final["Origin"].str.contains("Imports", case=False, na=False), "Imports", "NAS")
df_final['Target'] = np.where(df_final["Destination"].str.contains("Exports", case=False, na=False), "Exports", "DMI")

# to follow the same column structure, trade_type is deleted from the df
df_final = df_final.drop(columns=['dist_band'])

In [22]:
# save output
output_file = f"Oahu_MFA2018-2023.xlsx"
df_final.to_excel(output_file, index=False)

# save individually
for year, subset in df_final.groupby('Year'):
    filename = f"C:/Users/linsmasa/ODYM-master/revised_model/Imports_Exports/Oahu_MFA{year}.xlsx"
    subset.to_excel(filename, index=False)
    print(f"Saved {filename}")

Saved C:/Users/linsmasa/ODYM-master/revised_model/Imports_Exports/Oahu_MFA2018.xlsx
Saved C:/Users/linsmasa/ODYM-master/revised_model/Imports_Exports/Oahu_MFA2019.xlsx
Saved C:/Users/linsmasa/ODYM-master/revised_model/Imports_Exports/Oahu_MFA2020.xlsx
Saved C:/Users/linsmasa/ODYM-master/revised_model/Imports_Exports/Oahu_MFA2021.xlsx
Saved C:/Users/linsmasa/ODYM-master/revised_model/Imports_Exports/Oahu_MFA2022.xlsx
Saved C:/Users/linsmasa/ODYM-master/revised_model/Imports_Exports/Oahu_MFA2023.xlsx


#### 2) Available Data for Hawaii and Approximation of Oahu   
There is no data available for Oahu for years prior to 2013. Therefore, available statelevel data is used. To estimate the share of Oahu, we compare Hawaii-level data with Oahu-level data for the overlapping available time period of 2013-2022. As a second step, we calculate shares per sctg2 category and apply them on the historic Hawaii data on 1997-2012.

##### 2.1) Analysis of Hawaii data  
__2013-2017__

In [26]:
years = [2013, 2014, 2015, 2016, 2017]

In [28]:
# Dictionary for description 
desc_map = FAF4_sctg2.set_index("Code")["Commodity Description"]
mat_map = FAF4_sctg2.set_index("Code")["Material"]

In [30]:
for year in years:
    file_path = f"C:/Users/linsmasa/ODYM-master/revised_model/Imports_Exports/FAF4.5.1_State_{year}.csv"
    df = pd.read_csv(file_path)
    
    # Remove columns 11-13 (0-based index, so it's columns 10-13)
    df = df.drop(df.columns[10:13], axis=1)
    
    df = df.rename(columns={'dms_origst':'dms_orig','dms_destst':'dms_dest'})
    
    # Filter for rows where dms_orig or dms_dest is 151 (Honolulu)
    origin15 = df[df['dms_orig'] == 15]
    dest15 = df[(df['dms_dest'] == 15) & (df['dms_orig'] != 15)]
    
    # Combine the filtered datasets
    Hawaii = pd.concat([origin15, dest15], ignore_index=True)
    
    # Python reads some columns not as numbers
    Hawaii['dms_orig'] = pd.to_numeric(Hawaii['dms_orig'], errors='coerce')
    Hawaii['dms_dest'] = pd.to_numeric(Hawaii['dms_dest'], errors='coerce')
    Hawaii['fr_dest'] = Hawaii['fr_dest'].fillna(-1).astype(int)  # Replace NaN with -1, then convert to int
    Hawaii['fr_orig'] = Hawaii['fr_orig'].fillna(-1).astype(int)
    
    # Mapping function for locations
    def map_location(value):
        if value == -1:  # Skip processing if value is -1
            return -1
        if value == 15:
            return "Hawaii"
        elif 801 <= value <= 808:
            return {
                801: "Imports from Canada",
                802: "Imports from Mexico",
                803: "Imports from Rest of Americas",
                804: "Imports from Europe",
                805: "Imports from Africa",
                806: "Imports from SW & Central Asia",
                807: "Imports from Eastern Asia",
                808: "Imports from SE Asia & Oceania"
            }.get(value, value)
        else:
            return "Imports from US mainland"
            
    Hawaii['dms_orig'] = Hawaii['dms_orig'].apply(map_location)
    Hawaii['fr_orig'] = Hawaii['fr_orig'].apply(map_location)
    
    def map_location(value):
        if value == -1:  # Skip processing if value is -1
            return -1
        if value == 15:
            return "Hawaii"
        elif 801 <= value <= 808:
            return {
                801: "Exports to Canada",
                802: "Exports to Mexico",
                803: "Exports to Rest of Americas",
                804: "Exports to Europe",
                805: "Exports to Africa",
                806: "Exports to SW & Central Asia",
                807: "Exports to Eastern Asia",
                808: "Exports to SE Asia & Oceania"
            }.get(value, value)
        else:
            return "Exports to US mainland"
            
    Hawaii['dms_dest'] = Hawaii['dms_dest'].apply(map_location)
    Hawaii['fr_dest'] = Hawaii['fr_dest'].apply(map_location)
    
    # Define filtering options
    part1 = Hawaii[(Hawaii['fr_orig'] == -1) & (Hawaii['dms_orig'] == "Hawaii") & (Hawaii['dms_dest'] != "Hawaii") & (Hawaii['fr_dest'] == -1)].drop(columns=['fr_orig', 'fr_dest']).rename(columns={'dms_orig': 'Source', 'dms_dest': 'Target'})
    part2 = Hawaii[(Hawaii['fr_orig'] == -1) & (Hawaii['dms_orig'] == "Hawaii") & (Hawaii['dms_dest'] != "Hawaii") & (Hawaii['fr_dest'] != -1)].drop(columns=['fr_orig', 'fr_dest']).rename(columns={'dms_orig': 'Source', 'dms_dest': 'Target'})
    part3 = Hawaii[(Hawaii['fr_orig'] == -1) & (Hawaii['dms_orig'] == "Hawaii") & (Hawaii['dms_dest'] == "Hawaii") & (Hawaii['fr_dest'] != -1)].drop(columns=['fr_orig', 'dms_orig']).rename(columns={'dms_dest': 'Source', 'fr_dest': 'Target'})
    part4 = Hawaii[(Hawaii['fr_orig'] != -1) & (Hawaii['dms_orig'] == "Hawaii") & (Hawaii['dms_dest'] == "Hawaii") & (Hawaii['fr_dest'] == -1)].drop(columns=['dms_dest', 'fr_dest']).rename(columns={'fr_orig': 'Source', 'dms_orig': 'Target'})
    part5 = Hawaii[(Hawaii['fr_orig'] == -1) & (Hawaii['dms_orig'] != "Hawaii") & (Hawaii['dms_dest'] == "Hawaii") & (Hawaii['fr_dest'] == -1)].drop(columns=['fr_orig', 'fr_dest']).rename(columns={'dms_orig': 'Source', 'dms_dest': 'Target'})
    part6 = Hawaii[(Hawaii['fr_orig'] != -1) & (Hawaii['dms_orig'] != "Hawaii") & (Hawaii['dms_dest'] == "Hawaii") & (Hawaii['fr_dest'] == -1)].drop(columns=['fr_orig', 'fr_dest']).rename(columns={'dms_orig': 'Source', 'dms_dest': 'Target'})
    #revision
    part7 = Hawaii[(Hawaii['fr_orig'] == -1) & (Hawaii['dms_orig'] == "Hawaii") & (Hawaii['dms_dest'] == "Hawaii") & (Hawaii['fr_dest'] == -1)].drop(columns=['fr_orig', 'fr_dest']).rename(columns={'dms_orig': 'Source', 'dms_dest': 'Target'})
    
    # Combine all parts
    df_final = pd.concat([part1, part2, part3, part4, part5, part6, part7], ignore_index=True)
    
    # Rename columns
    df_final.columns = ["Source", "Target", "fr_inmode", "dms_mode", "fr_outmode", "sctg2", "trade_type", "Tons"]
    
    df_final["Commodity Description"] = df_final["sctg2"].map(desc_map)
    df_final["Material"] = df_final["sctg2"].map(mat_map)

    #here, Origin and Destination as well as Source and Target need to be adapted
    df_final = df_final.rename(columns={"Source": "Origin", "Target": "Destination"})
    df_final['Source'] = np.where(df_final["Origin"].str.contains("Imports", case=False, na=False), "Imports", "NAS")
    df_final['Target'] = np.where(df_final["Destination"].str.contains("Exports", case=False, na=False), "Exports", "DMI")
    df_final["Source_Index"] = np.where(df_final["Source"].str.contains("Imports", case=False, na=False), 0, 5) # 0 Imports, 5 NAS > Exports
    df_final["Target_Index"] = np.where(df_final["Target"].str.contains("Exports", case=False, na=False), 11, 1) # 11 Exports, 1 DMI
    df_final["Flow"] = "F" + df_final["Source_Index"].astype(str) + "_" + df_final["Target_Index"].astype(str)
    
    # Save results
    output_file = f"C:/Users/linsmasa/ODYM-master/revised_model/Imports_Exports/Hawaii_MFA{year}.xlsx"
    df_final.to_excel(output_file, index=False)
    print(f"Processed {year} and saved to {output_file}")

Processed 2013 and saved to C:/Users/linsmasa/ODYM-master/revised_model/Imports_Exports/Hawaii_MFA2013.xlsx
Processed 2014 and saved to C:/Users/linsmasa/ODYM-master/revised_model/Imports_Exports/Hawaii_MFA2014.xlsx
Processed 2015 and saved to C:/Users/linsmasa/ODYM-master/revised_model/Imports_Exports/Hawaii_MFA2015.xlsx
Processed 2016 and saved to C:/Users/linsmasa/ODYM-master/revised_model/Imports_Exports/Hawaii_MFA2016.xlsx
Processed 2017 and saved to C:/Users/linsmasa/ODYM-master/revised_model/Imports_Exports/Hawaii_MFA2017.xlsx


__2018-2023__

In [32]:
Hawaii = pd.read_csv("C:/Users/linsmasa/ODYM-master/revised_model/Imports_Exports/FAF5.6.1_State_2018-2023.csv")

In [34]:
desc_map = FAF5_sctg2.set_index("Code")["Commodity Description"]
mat_map = FAF5_sctg2.set_index("Code")["Material"]

In [36]:
# Keep only the first 15 columns (removing columns 17-30) (reduce the size of df)
Hawaii = Hawaii.iloc[:, :16]
df = Hawaii
# Filter for rows where dms_orig or dms_dest is 15 (Hawaii)
origin15 = df[df['dms_origst'] == 15]
dest15 = df[df['dms_destst'] == 15]

# Remove rows where dms_orig == dms_dest (both 15), is only deleted in one of them, so there are no duplicates
dest15 = dest15[dest15['dms_origst'] != 15]

# Combine filtered data
Hawaii = pd.concat([origin15, dest15], ignore_index=True) 

# Python reads some columns not as numbers
Hawaii['dms_origst'] = pd.to_numeric(Hawaii['dms_origst'], errors='coerce')
Hawaii['dms_destst'] = pd.to_numeric(Hawaii['dms_destst'], errors='coerce')
Hawaii['fr_dest'] = Hawaii['fr_dest'].fillna(-1).astype(int)  # Replace NaN with -1, then convert to int
Hawaii['fr_orig'] = Hawaii['fr_orig'].fillna(-1).astype(int)

# Replace numeric locations with names, first dms_origst & fr_orig
def map_location(value):
    if value == -1:  # Skip processing if value is -1
        return -1
    if value == 15:
        return "Hawaii"
    elif 801 <= value <= 808:
        return {
            801: "Imports from Canada",
            802: "Imports from Mexico",
            803: "Imports from Rest of Americas",
            804: "Imports from Europe",
            805: "Imports from Africa",
            806: "Imports from SW & Central Asia",
            807: "Imports from Eastern Asia",
            808: "Imports from SE Asia & Oceania"
        }.get(value, value)
    else:
        return "Imports from US mainland"

Hawaii['dms_origst'] = Hawaii['dms_origst'].apply(map_location)
Hawaii['fr_orig'] = Hawaii['fr_orig'].apply(map_location)

def map_location(value):
    if value == -1:  # Skip processing if value is -1
        return -1
    if value == 15:
        return "Hawaii"
    elif 801 <= value <= 808:
        return {
            801: "Exports to Canada",
            802: "Exports to Mexico",
            803: "Exports to Rest of Americas",
            804: "Exports to Europe",
            805: "Exports to Africa",
            806: "Exports to SW & Central Asia",
            807: "Exports to Eastern Asia",
            808: "Exports to SE Asia & Oceania"
        }.get(value, value)
    else:
        return "Exports to US mainland"

Hawaii['dms_destst'] = Hawaii['dms_destst'].apply(map_location)
Hawaii['fr_dest'] = Hawaii['fr_dest'].apply(map_location)

part1 = Hawaii[(Hawaii['fr_orig'] == -1) & (Hawaii['dms_origst'] == "Hawaii") & (Hawaii['dms_destst'] != "Hawaii") & (Hawaii['fr_dest'] == -1)].drop(columns=['fr_orig', 'fr_dest']).rename(columns={'dms_origst': 'Source', 'dms_destst': 'Target'})
part2 = Hawaii[(Hawaii['fr_orig'] == -1) & (Hawaii['dms_origst'] == "Hawaii") & (Hawaii['dms_destst'] != "Hawaii") & (Hawaii['fr_dest'] != -1)].drop(columns=['fr_orig', 'fr_dest']).rename(columns={'dms_origst': 'Source', 'dms_destst': 'Target'})
part3 = Hawaii[(Hawaii['fr_orig'] == -1) & (Hawaii['dms_origst'] == "Hawaii") & (Hawaii['dms_destst'] == "Hawaii") & (Hawaii['fr_dest'] != -1)].drop(columns=['fr_orig', 'dms_origst']).rename(columns={'dms_destst': 'Source', 'fr_dest': 'Target'})
part4 = Hawaii[(Hawaii['fr_orig'] != -1) & (Hawaii['dms_origst'] == "Hawaii") & (Hawaii['dms_destst'] == "Hawaii") & (Hawaii['fr_dest'] == -1)].drop(columns=['dms_destst', 'fr_dest']).rename(columns={'fr_orig': 'Source', 'dms_origst': 'Target'})
part5 = Hawaii[(Hawaii['fr_orig'] == -1) & (Hawaii['dms_origst'] != "Hawaii") & (Hawaii['dms_destst'] == "Hawaii") & (Hawaii['fr_dest'] == -1)].drop(columns=['fr_orig', 'fr_dest']).rename(columns={'dms_origst': 'Source', 'dms_destst': 'Target'})
part6 = Hawaii[(Hawaii['fr_orig'] != -1) & (Hawaii['dms_origst'] != "Hawaii") & (Hawaii['dms_destst'] == "Hawaii") & (Hawaii['fr_dest'] == -1)].drop(columns=['fr_orig', 'fr_dest']).rename(columns={'dms_origst': 'Source', 'dms_destst': 'Target'})
#revision
part7 = Hawaii[(Hawaii['fr_orig'] == -1) & (Hawaii['dms_origst'] == "Hawaii") & (Hawaii['dms_destst'] == "Hawaii") & (Hawaii['fr_dest'] == -1)].drop(columns=['fr_orig', 'fr_dest']).rename(columns={'dms_origst': 'Source', 'dms_destst': 'Target'})
    
# Combine all parts
df_final = pd.concat([part1, part2, part3, part4, part5, part6, part7], ignore_index=True)

# Map SCTG product codes to product names
df_final["Commodity Description"] = df_final["sctg2"].map(desc_map)
df_final["Material"] = df_final["sctg2"].map(mat_map)

In [38]:
# change the format from wide to long
# Automatically detect year columns (those starting with 'tons_')
year_cols = [c for c in df_final.columns if c.startswith("tons_")]

# Melt while keeping all other columns
df_long = df_final.melt(
    id_vars=[c for c in df_final.columns if c not in year_cols],
    value_vars=year_cols,
    var_name='Year',
    value_name='Tons'
)

# Extract numeric year from 'kg_2018' → 2018
df_long['Year'] = df_long['Year'].str.extract(r'(\d{4})').astype(int)

df_final = df_long.copy()
df_final = df_final.rename(columns={"Source": "Origin", "Target": "Destination"})
df_final['Source'] = np.where(df_final["Origin"].str.contains("Imports", case=False, na=False), "Imports", "NAS")
df_final['Target'] = np.where(df_final["Destination"].str.contains("Exports", case=False, na=False), "Exports", "DMI")

# to follow the same column structure, trade_type is deleted from the df
df_final = df_final.drop(columns=['dist_band'])

In [40]:
# save output
output_file = f"Hawaii_MFA2018-2023.xlsx"
df_final.to_excel(output_file, index=False)

# save individually
for year, subset in df_final.groupby('Year'):
    filename = f"C:/Users/linsmasa/ODYM-master/revised_model/Imports_Exports/Hawaii_MFA{year}.xlsx"
    subset.to_excel(filename, index=False)
    print(f"Saved {filename}")

Saved C:/Users/linsmasa/ODYM-master/revised_model/Imports_Exports/Hawaii_MFA2018.xlsx
Saved C:/Users/linsmasa/ODYM-master/revised_model/Imports_Exports/Hawaii_MFA2019.xlsx
Saved C:/Users/linsmasa/ODYM-master/revised_model/Imports_Exports/Hawaii_MFA2020.xlsx
Saved C:/Users/linsmasa/ODYM-master/revised_model/Imports_Exports/Hawaii_MFA2021.xlsx
Saved C:/Users/linsmasa/ODYM-master/revised_model/Imports_Exports/Hawaii_MFA2022.xlsx
Saved C:/Users/linsmasa/ODYM-master/revised_model/Imports_Exports/Hawaii_MFA2023.xlsx


#### 3) Comparison and Creation of a share dataframe

#### Revised Method  
* Calculate a share of Oahu of domestic trade flows (within Hawaii)
* Calculate a share of Oahu of foreign trade flows 

In [46]:
# create a loop for all the years
years = range(2013, 2024)
results = {}  #store results in a dictionary

In [48]:
for year in years:
    # Read the data for the current year
    Oahu = pd.read_excel(f"C:/Users/linsmasa/ODYM-master/revised_model/Imports_Exports/Oahu_MFA{year}.xlsx")
    Hawaii = pd.read_excel(f"C:/Users/linsmasa/ODYM-master/revised_model/Imports_Exports/Hawaii_MFA{year}.xlsx")

    #foreign trade flows
    expO = (
    Oahu[
        (Oahu["Origin"] == "Honolulu") &
        (Oahu["Destination"] != "Exports to Rest of HI")
    ]
    .groupby("Material")["Tons"]
    .sum()
    .to_dict()
    )
    expH = (
    Hawaii[
        (Hawaii["Origin"] == "Hawaii") &
        (Hawaii["Destination"] != "Hawaii")
    ]
    .groupby("Material")["Tons"]
    .sum()
    .to_dict()
    )
    impO = (
    Oahu[
        (Oahu["Origin"] != "Honolulu") & (Oahu["Origin"] != "Imports from Rest of HI") &
        (Oahu["Destination"] == "Honolulu")
    ]
    .groupby("Material")["Tons"]
    .sum()
    .to_dict()
    )
    impH = (
    Hawaii[
        (Hawaii["Origin"] != "Hawaii") &
        (Hawaii["Destination"] == "Hawaii")
    ]
    .groupby("Material")["Tons"]
    .sum()
    .to_dict()
    )
    #internal trade flows
    IimpO = (
    Oahu[
        (Oahu["Origin"] == "Imports from Rest of HI") &
        (Oahu["Destination"] == "Honolulu")
    ]
    .groupby("Material")["Tons"]
    .sum()
    .to_dict()
    )
    IexpO = (
    Oahu[
        (Oahu["Origin"] == "Honolulu") &
        (Oahu["Destination"] == "Exports to Rest of HI")
    ]
    .groupby("Material")["Tons"]
    .sum()
    .to_dict()
    )
    tradeH = (
    Hawaii[
        (Hawaii["Origin"] == "Hawaii") &
        (Hawaii["Destination"] == "Hawaii")
    ]
    .groupby("Material")["Tons"]
    .sum()
    .to_dict()
    )
    share = pd.DataFrame({
    "expO": expO,
    "expH": expH,
    "impO": impO,
    "impH": impH,
    "IimpO": IimpO,
    "IexpO": IexpO,
    "tradeH": tradeH
    })

    share["foreign export share Oahu"] = share["expO"]/share["expH"]
    share["foreign import share Oahu"] = share["impO"]/share["impH"]
    share["domestic export share Oahu"] = share["IexpO"]/share["tradeH"]
    share["domestic import share Oahu"] = share["IimpO"]/share["tradeH"]
    # Store the DataFrame for this year in the results dictionary
    results[year] = share

In [49]:
# material specific shares
# empty DataFrame to store the aggregate data
aggregate_data = pd.DataFrame()

# Loop over the years and append data to the aggregate DataFrame
for year in years:
    share = results[year]  # Get the DataFrame for each year
    share['Year'] = year  # Add a 'Year' column to track the year

    # Append the data to the aggregate_data DataFrame
    aggregate_data = pd.concat([aggregate_data, share])

In [52]:
aggregate_data

,expO,expH,impO,impH,IimpO,IexpO,tradeH,foreign export share Oahu,foreign import share Oahu,domestic export share Oahu,domestic import share Oahu,Year
chemicals,5.679800,10.268500,53.637900,76.064400,0.546900,10.703400,1580.626400,0.553128,0.705164,0.006772,0.000346,2013
construction material and minerals,21.996300,23.001500,658.656700,956.372900,199.264800,55.240800,13093.926700,0.956299,0.688703,0.004219,0.015218,2013
electronics,9.461700,11.466900,40.917200,56.482100,0.918500,1.779300,49.372100,0.825131,0.724428,0.036039,0.018604,2013
final products,60.082000,77.001500,215.704000,270.511200,4.374200,27.024800,293.720400,0.780271,0.797394,0.092009,0.014892,2013
food,35.232300,84.017000,476.482800,600.662600,78.428500,189.585600,1973.606400,0.419347,0.793262,0.096060,0.039739,2013
...,...,...,...,...,...,...,...,...,...,...,...,...
motorized vehicles,4.616230,6.107709,105.343908,136.981611,0.155840,18.315061,85.178065,0.755804,0.769037,0.215021,0.001830,2023
paper,2.965951,3.469701,67.628349,97.252034,1.411843,11.487459,174.986449,0.854815,0.695393,0.065648,0.008068,2023
plastic,0.364488,0.493241,57.746381,128.228439,33.878444,15.259921,133.061707,0.738965,0.450340,0.114683,0.254607,2023
waste,39.281433,56.576334,0.036707,0.052107,NaN,2.298755,3391.457168,0.694309,0.704454,0.000678,NaN,2023


In [54]:
aggregate_data = aggregate_data.reset_index(drop=False).rename(columns={"index": "Material"})
aggregate_data.head()

,Material,expO,expH,impO,impH,IimpO,IexpO,tradeH,foreign export share Oahu,foreign import share Oahu,domestic export share Oahu,domestic import share Oahu,Year
0,chemicals,5.6798,10.2685,53.6379,76.0644,0.5469,10.7034,1580.6264,0.553128,0.705164,0.006772,0.000346,2013
1,construction material and minerals,21.9963,23.0015,658.6567,956.3729,199.2648,55.2408,13093.9267,0.956299,0.688703,0.004219,0.015218,2013
2,electronics,9.4617,11.4669,40.9172,56.4821,0.9185,1.7793,49.3721,0.825131,0.724428,0.036039,0.018604,2013
3,final products,60.0820,77.0015,215.7040,270.5112,4.3742,27.0248,293.7204,0.780271,0.797394,0.092009,0.014892,2013
4,food,35.2323,84.0170,476.4828,600.6626,78.4285,189.5856,1973.6064,0.419347,0.793262,0.096060,0.039739,2013


In [56]:
aggregate_data.to_excel("C:/Users/linsmasa/ODYM-master/revised_model/Imports_Exports/shares_by_material.xlsx", index = False)

### Application of shares

In [58]:
# Now, group by material and calculate the mean for 'Import_Oahu_Share' and 'Export_Hawaii_Share' across all years
mean_shares_by_material = aggregate_data.groupby('Material',as_index=False)[['foreign export share Oahu', 'foreign import share Oahu', 'domestic export share Oahu', 'domestic import share Oahu']].mean()
#mean_shares_by_material.loc[mean_shares_by_material['Material'] == 'waste', 'Import_Oahu_Share'] = 1
# Display the result
print(mean_shares_by_material)

                              Material  foreign export share Oahu  \
0                            chemicals                   0.629051   
1   construction material and minerals                   0.954345   
2                          electronics                   0.813222   
3                       final products                   0.787652   
4                                 food                   0.376267   
5                         fossil fuels                   0.770789   
6                               metals                   0.925358   
7                        mixed freight                   0.828600   
8                   motorized vehicles                   0.779924   
9                                paper                   0.904934   
10                             plastic                   0.860229   
11                               waste                   0.703412   
12                                wood                   0.679911   

    foreign import share Oahu  do

In [62]:
# Save the DataFrame with 'Material' as a regular column
mean_shares_by_material.to_excel("disaggregation_parameters.xlsx", index=False)

In [64]:
#read the data
Hawaii = pd.read_csv("FAF5.5.1_Reprocessed_1997-2012_State.csv")
Hawaii = Hawaii.iloc[:, :13] # Keep only the first 13 columns (removing columns 16-30) (reduce the size of df)
desc_map = FAF5_sctg2.set_index("Code")["Commodity Description"]
mat_map = FAF5_sctg2.set_index("Code")["Material"]
Hawaii.head()

,fr_orig,dms_origst,dms_destst,fr_dest,fr_inmode,dms_mode,fr_outmode,sctg2,trade_type,tons_1997,tons_2002,tons_2007,tons_2012
0,NaN,1,1,NaN,NaN,1,NaN,1,1,584.804465,646.482836,1219.469158,3084.7678
1,NaN,1,5,NaN,NaN,1,NaN,1,1,23.871883,0.564601,0.000000,0.0000
2,NaN,1,5,NaN,NaN,2,NaN,1,1,0.767310,0.000000,0.000000,0.0000
3,NaN,1,8,NaN,NaN,1,NaN,1,1,0.344100,0.000000,0.000000,0.0000
4,NaN,1,8,NaN,NaN,2,NaN,1,1,0.004000,0.000000,0.000000,0.0000


In [66]:
df = Hawaii
# Filter for rows where dms_orig or dms_dest is 15 (Hawaii)
origin15 = df[df['dms_origst'] == 15]
dest15 = df[df['dms_destst'] == 15]

# Remove rows where dms_orig == dms_dest (both 15), is only deleted in one of them, so there are no duplicates
dest15 = dest15[dest15['dms_origst'] != 15]

# Combine filtered data
Hawaii = pd.concat([origin15, dest15], ignore_index=True) 
Hawaii['dms_origst'] = pd.to_numeric(Hawaii['dms_origst'], errors='coerce')
Hawaii['dms_destst'] = pd.to_numeric(Hawaii['dms_destst'], errors='coerce')
Hawaii['fr_dest'] = Hawaii['fr_dest'].fillna(-1).astype(int)  # Replace NaN with -1, then convert to int
Hawaii['fr_orig'] = Hawaii['fr_orig'].fillna(-1).astype(int)

Hawaii.head()

,fr_orig,dms_origst,dms_destst,fr_dest,fr_inmode,dms_mode,fr_outmode,sctg2,trade_type,tons_1997,tons_2002,tons_2007,tons_2012
0,-1,15,6,-1,NaN,5,NaN,1,1,0.000000,0.000000,0.973401,0.0000
1,-1,15,12,-1,NaN,5,NaN,1,1,0.000000,0.000000,0.006300,0.0000
2,-1,15,15,-1,NaN,1,NaN,1,1,60.761229,64.488714,42.552071,40.8731
3,-1,15,15,-1,NaN,4,NaN,1,1,0.000000,0.000000,0.006911,0.0000
4,-1,15,15,-1,NaN,1,NaN,2,1,24.874932,27.736798,7.472079,24.1710


In [68]:
# Replace numeric locations with names, first dms_origst & fr_orig
def map_location(value):
    if value == -1:  # Skip processing if value is -1
        return -1
    if value == 15:
        return "Hawaii"
    elif 801 <= value <= 808:
        return {
            801: "Imports from Canada",
            802: "Imports from Mexico",
            803: "Imports from Rest of Americas",
            804: "Imports from Europe",
            805: "Imports from Africa",
            806: "Imports from SW & Central Asia",
            807: "Imports from Eastern Asia",
            808: "Imports from SE Asia & Oceania"
        }.get(value, value)
    else:
        return "Imports from US mainland"

Hawaii['dms_origst'] = Hawaii['dms_origst'].apply(map_location)
Hawaii['fr_orig'] = Hawaii['fr_orig'].apply(map_location)

def map_location(value):
    if value == -1:  # Skip processing if value is -1
        return -1
    if value == 15:
        return "Hawaii"
    elif 801 <= value <= 808:
        return {
            801: "Exports to Canada",
            802: "Exports to Mexico",
            803: "Exports to Rest of Americas",
            804: "Exports to Europe",
            805: "Exports to Africa",
            806: "Exports to SW & Central Asia",
            807: "Exports to Eastern Asia",
            808: "Exports to SE Asia & Oceania"
        }.get(value, value)
    else:
        return "Exports to US mainland"

Hawaii['dms_destst'] = Hawaii['dms_destst'].apply(map_location)
Hawaii['fr_dest'] = Hawaii['fr_dest'].apply(map_location)
Hawaii.head()

,fr_orig,dms_origst,dms_destst,fr_dest,fr_inmode,dms_mode,fr_outmode,sctg2,trade_type,tons_1997,tons_2002,tons_2007,tons_2012
0,-1,Hawaii,Exports to US mainland,-1,NaN,5,NaN,1,1,0.000000,0.000000,0.973401,0.0000
1,-1,Hawaii,Exports to US mainland,-1,NaN,5,NaN,1,1,0.000000,0.000000,0.006300,0.0000
2,-1,Hawaii,Hawaii,-1,NaN,1,NaN,1,1,60.761229,64.488714,42.552071,40.8731
3,-1,Hawaii,Hawaii,-1,NaN,4,NaN,1,1,0.000000,0.000000,0.006911,0.0000
4,-1,Hawaii,Hawaii,-1,NaN,1,NaN,2,1,24.874932,27.736798,7.472079,24.1710


In [70]:
part1 = Hawaii[(Hawaii['fr_orig'] == -1) & (Hawaii['dms_origst'] == "Hawaii") & (Hawaii['dms_destst'] != "Hawaii") & (Hawaii['fr_dest'] == -1)].drop(columns=['fr_orig', 'fr_dest']).rename(columns={'dms_origst': 'Source', 'dms_destst': 'Target'})
part2 = Hawaii[(Hawaii['fr_orig'] == -1) & (Hawaii['dms_origst'] == "Hawaii") & (Hawaii['dms_destst'] != "Hawaii") & (Hawaii['fr_dest'] != -1)].drop(columns=['fr_orig', 'fr_dest']).rename(columns={'dms_origst': 'Source', 'dms_destst': 'Target'})
part3 = Hawaii[(Hawaii['fr_orig'] == -1) & (Hawaii['dms_origst'] == "Hawaii") & (Hawaii['dms_destst'] == "Hawaii") & (Hawaii['fr_dest'] != -1)].drop(columns=['fr_orig', 'dms_origst']).rename(columns={'dms_destst': 'Source', 'fr_dest': 'Target'})
part4 = Hawaii[(Hawaii['fr_orig'] != -1) & (Hawaii['dms_origst'] == "Hawaii") & (Hawaii['dms_destst'] == "Hawaii") & (Hawaii['fr_dest'] == -1)].drop(columns=['dms_destst', 'fr_dest']).rename(columns={'fr_orig': 'Source', 'dms_origst': 'Target'})
part5 = Hawaii[(Hawaii['fr_orig'] == -1) & (Hawaii['dms_origst'] != "Hawaii") & (Hawaii['dms_destst'] == "Hawaii") & (Hawaii['fr_dest'] == -1)].drop(columns=['fr_orig', 'fr_dest']).rename(columns={'dms_origst': 'Source', 'dms_destst': 'Target'})
part6 = Hawaii[(Hawaii['fr_orig'] != -1) & (Hawaii['dms_origst'] != "Hawaii") & (Hawaii['dms_destst'] == "Hawaii") & (Hawaii['fr_dest'] == -1)].drop(columns=['fr_orig', 'fr_dest']).rename(columns={'dms_origst': 'Source', 'dms_destst': 'Target'})
#revision
part7 = Hawaii[(Hawaii['fr_orig'] == -1) & (Hawaii['dms_origst'] == "Hawaii") & (Hawaii['dms_destst'] == "Hawaii") & (Hawaii['fr_dest'] == -1)].drop(columns=['fr_orig', 'fr_dest']).rename(columns={'dms_origst': 'Source', 'dms_destst': 'Target'})
# Combine all parts
df_final = pd.concat([part1, part2, part3, part4, part5, part6, part7], ignore_index=True)
df_final.head()

,Source,Target,fr_inmode,dms_mode,fr_outmode,sctg2,trade_type,tons_1997,tons_2002,tons_2007,tons_2012
0,Hawaii,Exports to US mainland,NaN,5,NaN,1,1,0.000,0.000,0.973401,0.0000
1,Hawaii,Exports to US mainland,NaN,5,NaN,1,1,0.000,0.000,0.006300,0.0000
2,Hawaii,Exports to US mainland,NaN,4,NaN,3,1,0.000,0.000,0.000000,0.0744
3,Hawaii,Exports to US mainland,NaN,5,NaN,3,1,0.000,0.000,0.000000,0.2316
4,Hawaii,Exports to US mainland,NaN,5,NaN,3,1,0.002,0.856,0.011100,0.1490


In [72]:
# Map SCTG product codes to product names
df_final["Commodity Description"] = df_final["sctg2"].map(desc_map)
df_final["Material"] = df_final["sctg2"].map(mat_map)

In [74]:
#here, Origin and Destination as well as Source and Target need to be adapted
df_final = df_final.rename(columns={"Source": "Origin", "Target": "Destination"})
df_final['Source'] = np.where(df_final["Origin"].str.contains("Imports", case=False, na=False), "Imports", "NAS")
df_final['Target'] = np.where(df_final["Destination"].str.contains("Exports", case=False, na=False), "Exports", "DMI")

In [76]:
df_final.head()

,Origin,Destination,fr_inmode,dms_mode,fr_outmode,sctg2,trade_type,tons_1997,tons_2002,tons_2007,tons_2012,Commodity Description,Material,Source,Target
0,Hawaii,Exports to US mainland,NaN,5,NaN,1,1,0.000,0.000,0.973401,0.0000,Live animals/fish,food,NAS,Exports
1,Hawaii,Exports to US mainland,NaN,5,NaN,1,1,0.000,0.000,0.006300,0.0000,Live animals/fish,food,NAS,Exports
2,Hawaii,Exports to US mainland,NaN,4,NaN,3,1,0.000,0.000,0.000000,0.0744,Other ag prods.,food,NAS,Exports
3,Hawaii,Exports to US mainland,NaN,5,NaN,3,1,0.000,0.000,0.000000,0.2316,Other ag prods.,food,NAS,Exports
4,Hawaii,Exports to US mainland,NaN,5,NaN,3,1,0.002,0.856,0.011100,0.1490,Other ag prods.,food,NAS,Exports


In [78]:
df_final.to_excel("Hawaii_MFA1997-2012.xlsx", index=False)

In [83]:
parameters = pd.read_excel("disaggregation_parameters.xlsx")
years = (1997, 2002, 2007, 2012)

In [85]:
all_years = []

for year in years:

    df = pd.read_excel("C:/Users/linsmasa/ODYM-master/revised_model/Hawaii_MFA1997-2012.xlsx")

    # Merge dataframe with parameters
    merged_df = pd.merge(df, parameters, on="Material", how="left")
    print(merged_df.columns.tolist())

    # Condition 1: Hawaii -> Foreign
    condition1 = (
        (merged_df["Origin"] == "Hawaii") &
        (merged_df["Destination"] != "Hawaii")
    )

    # Condition 2: Foreign -> Honolulu
    condition2 = (
        (merged_df["Origin"] != "Hawaii") &
        (merged_df["Destination"] == "Hawaii")
    )

    # Condition 3: Hawaii -> Hawaii
    condition3 = (
        (merged_df["Origin"] == "Hawaii") &
        (merged_df["Destination"] == "Hawaii")
    )

    # Create separate dataframes
    df1 = merged_df.loc[condition1].copy()
    df2 = merged_df.loc[condition2].copy()
    df3 = merged_df.loc[condition3].copy()

    # Calculate Tons for each condition
    df1["Tons"] = (
        df1[f"tons_{year}"] * df1["foreign export share Oahu"]
    )

    df2["Tons"] = (
        df2[f"tons_{year}"] * df2["foreign import share Oahu"]
    )

    # Domestic import
    df3_import = df3.copy()
    df3_import["Tons"] = (
        df3_import[f"tons_{year}"] *
        df3_import["domestic import share Oahu"]
    )
    df3_import["Source"] = "Imports"
    df3_import["Target"] = "DMI"


    # Domestic export
    df3_export = df3.copy()
    df3_export["Tons"] = (
        df3_export[f"tons_{year}"] *
        df3_export["domestic export share Oahu"]
    )
    df3_export["Source"] = "DMI"
    df3_export["Target"] = "Exports"

    # Combine all results
    year_df = pd.concat(
        [df1, df2, df3_import, df3_export],
        ignore_index=True
    )

    # Add year
    year_df["Year"] = year
    print(year_df.columns.tolist())
    # Append to list
    all_years.append(year_df)


# Combine all years
final_df = pd.concat(all_years, ignore_index=True)

display(final_df)
final_df.to_excel(f"C:/Users/linsmasa/ODYM-master/revised_model/Imports_Exports/new_calculation_imp_exp.xlsx", index=False)

['Origin', 'Destination', 'fr_inmode', 'dms_mode', 'fr_outmode', 'sctg2', 'trade_type', 'tons_1997', 'tons_2002', 'tons_2007', 'tons_2012', 'Commodity Description', 'Material', 'Source', 'Target', 'foreign export share Oahu', 'foreign import share Oahu', 'domestic export share Oahu', 'domestic import share Oahu']
['Origin', 'Destination', 'fr_inmode', 'dms_mode', 'fr_outmode', 'sctg2', 'trade_type', 'tons_1997', 'tons_2002', 'tons_2007', 'tons_2012', 'Commodity Description', 'Material', 'Source', 'Target', 'foreign export share Oahu', 'foreign import share Oahu', 'domestic export share Oahu', 'domestic import share Oahu', 'Tons', 'Year']
['Origin', 'Destination', 'fr_inmode', 'dms_mode', 'fr_outmode', 'sctg2', 'trade_type', 'tons_1997', 'tons_2002', 'tons_2007', 'tons_2012', 'Commodity Description', 'Material', 'Source', 'Target', 'foreign export share Oahu', 'foreign import share Oahu', 'domestic export share Oahu', 'domestic import share Oahu']
['Origin', 'Destination', 'fr_inmode', 

,Origin,Destination,fr_inmode,dms_mode,fr_outmode,sctg2,trade_type,tons_1997,tons_2002,tons_2007,...,Commodity Description,Material,Source,Target,foreign export share Oahu,foreign import share Oahu,domestic export share Oahu,domestic import share Oahu,Tons,Year
0,Hawaii,Exports to US mainland,NaN,5,NaN,1,1,0.000000,0.000000,0.973401,...,Live animals/fish,food,NAS,Exports,0.376267,0.798819,0.064687,0.026594,0.000000,1997
1,Hawaii,Exports to US mainland,NaN,5,NaN,1,1,0.000000,0.000000,0.006300,...,Live animals/fish,food,NAS,Exports,0.376267,0.798819,0.064687,0.026594,0.000000,1997
2,Hawaii,Exports to US mainland,NaN,4,NaN,3,1,0.000000,0.000000,0.000000,...,Other ag prods.,food,NAS,Exports,0.376267,0.798819,0.064687,0.026594,0.000000,1997
3,Hawaii,Exports to US mainland,NaN,5,NaN,3,1,0.000000,0.000000,0.000000,...,Other ag prods.,food,NAS,Exports,0.376267,0.798819,0.064687,0.026594,0.000000,1997
4,Hawaii,Exports to US mainland,NaN,5,NaN,3,1,0.002000,0.856000,0.011100,...,Other ag prods.,food,NAS,Exports,0.376267,0.798819,0.064687,0.026594,0.000753,1997
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46335,Hawaii,Hawaii,NaN,5,NaN,41,1,3.171041,0.000000,687.907857,...,Waste/scrap,waste,DMI,Exports,0.703412,0.703346,0.000808,0.013240,0.042022,2012
46336,Hawaii,Hawaii,NaN,1,NaN,43,1,103.019337,400.216612,456.271470,...,Mixed freight,mixed freight,DMI,Exports,0.828600,0.767605,0.263190,0.005804,76.946759,2012
46337,Hawaii,Hawaii,NaN,3,NaN,43,1,0.000000,0.026000,9.217208,...,Mixed freight,mixed freight,DMI,Exports,0.828600,0.767605,0.263190,0.005804,6.286328,2012
46338,Hawaii,Hawaii,NaN,4,NaN,43,1,0.260172,0.029000,13.033016,...,Mixed freight,mixed freight,DMI,Exports,0.828600,0.767605,0.263190,0.005804,2.370529,2012


In [86]:
final_df

,Origin,Destination,fr_inmode,dms_mode,fr_outmode,sctg2,trade_type,tons_1997,tons_2002,tons_2007,...,Commodity Description,Material,Source,Target,foreign export share Oahu,foreign import share Oahu,domestic export share Oahu,domestic import share Oahu,Tons,Year
0,Hawaii,Exports to US mainland,NaN,5,NaN,1,1,0.000000,0.000000,0.973401,...,Live animals/fish,food,NAS,Exports,0.376267,0.798819,0.064687,0.026594,0.000000,1997
1,Hawaii,Exports to US mainland,NaN,5,NaN,1,1,0.000000,0.000000,0.006300,...,Live animals/fish,food,NAS,Exports,0.376267,0.798819,0.064687,0.026594,0.000000,1997
2,Hawaii,Exports to US mainland,NaN,4,NaN,3,1,0.000000,0.000000,0.000000,...,Other ag prods.,food,NAS,Exports,0.376267,0.798819,0.064687,0.026594,0.000000,1997
3,Hawaii,Exports to US mainland,NaN,5,NaN,3,1,0.000000,0.000000,0.000000,...,Other ag prods.,food,NAS,Exports,0.376267,0.798819,0.064687,0.026594,0.000000,1997
4,Hawaii,Exports to US mainland,NaN,5,NaN,3,1,0.002000,0.856000,0.011100,...,Other ag prods.,food,NAS,Exports,0.376267,0.798819,0.064687,0.026594,0.000753,1997
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46335,Hawaii,Hawaii,NaN,5,NaN,41,1,3.171041,0.000000,687.907857,...,Waste/scrap,waste,DMI,Exports,0.703412,0.703346,0.000808,0.013240,0.042022,2012
46336,Hawaii,Hawaii,NaN,1,NaN,43,1,103.019337,400.216612,456.271470,...,Mixed freight,mixed freight,DMI,Exports,0.828600,0.767605,0.263190,0.005804,76.946759,2012
46337,Hawaii,Hawaii,NaN,3,NaN,43,1,0.000000,0.026000,9.217208,...,Mixed freight,mixed freight,DMI,Exports,0.828600,0.767605,0.263190,0.005804,6.286328,2012
46338,Hawaii,Hawaii,NaN,4,NaN,43,1,0.260172,0.029000,13.033016,...,Mixed freight,mixed freight,DMI,Exports,0.828600,0.767605,0.263190,0.005804,2.370529,2012


In [89]:
# Aggregate tons by specific combination
final_df[["Source", "Target"]] = final_df[["Source", "Target"]].replace("NAS", "DMI")
final_df = (
    final_df
    .groupby(
        ["Material", "Source", "Target", "Year", "Commodity Description"],
        as_index=False
    )["Tons"]
    .sum()
).rename(columns={'Tons': "kilotons"})

# Reorder columns
final_df = final_df[
    ["Material", "Commodity Description", "Source", "Target", "kilotons", "Year"]
]

display(final_df)

,Material,Commodity Description,Source,Target,kilotons,Year
0,chemicals,Basic chemicals,DMI,Exports,1.959304,1997
1,chemicals,Chemical prods.,DMI,Exports,3.077776,1997
2,chemicals,Fertilizers,DMI,Exports,1.939737,1997
3,chemicals,Pharmaceuticals,DMI,Exports,0.905373,1997
4,chemicals,Basic chemicals,DMI,Exports,1.233983,2002
...,...,...,...,...,...,...
331,wood,Wood prods.,Imports,DMI,188.226998,2002
332,wood,Logs,Imports,DMI,0.373343,2007
333,wood,Wood prods.,Imports,DMI,263.244530,2007
334,wood,Logs,Imports,DMI,9.816932,2012


In [91]:
final_df.to_excel(f"C:/Users/linsmasa/ODYM-master/revised_model/Imports_Exports/total_imp_exp_1997-2012_revised.xlsx", index=False)

### One big dataframe containing all import and export data

In [281]:
# load all the data
# df1997 = pd.read_excel("C:/Users/linsmasa/ODYM-master/revised_model/Imports_Exports/Oahu_MFA1997.xlsx")
# df2002 = pd.read_excel("C:/Users/linsmasa/ODYM-master/revised_model/Imports_Exports/Oahu_MFA2002.xlsx")
# df2007 = pd.read_excel("C:/Users/linsmasa/ODYM-master/revised_model/Imports_Exports/Oahu_MFA2007.xlsx")
# df2012 = pd.read_excel("C:/Users/linsmasa/ODYM-master/revised_model/Imports_Exports/Oahu_MFA2012.xlsx")
dfpre2013 = pd.read_excel("C:/Users/linsmasa/ODYM-master/revised_model/Imports_Exports/total_imp_exp_1997-2012_revised.xlsx")
df2017 = pd.read_excel("C:/Users/linsmasa/ODYM-master/revised_model/Imports_Exports/Oahu_MFA2017.xlsx")
df2022 = pd.read_excel("C:/Users/linsmasa/ODYM-master/revised_model/Imports_Exports/Oahu_MFA2022.xlsx")

In [193]:
df2017

,Origin,Destination,fr_inmode,dms_mode,fr_outmode,sctg2,trade_type,Tons,Commodity Description,Material,Source,Target,Year
0,Honolulu,Exports to Rest of HI,NaN,3,NaN,2,1,0.8484,Cereal Grains (includes seed),food,NAS,Exports,2017
1,Honolulu,Exports to US mainland,NaN,4,NaN,3,1,0.0001,"Agricultural Products (excludes Animal Feed, C...",food,NAS,Exports,2017
2,Honolulu,Exports to US mainland,NaN,4,NaN,3,1,0.0286,"Agricultural Products (excludes Animal Feed, C...",food,NAS,Exports,2017
3,Honolulu,Exports to US mainland,NaN,4,NaN,3,1,0.0001,"Agricultural Products (excludes Animal Feed, C...",food,NAS,Exports,2017
4,Honolulu,Exports to US mainland,NaN,4,NaN,3,1,0.0001,"Agricultural Products (excludes Animal Feed, C...",food,NAS,Exports,2017
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4736,Imports from SE Asia & Oceania,Honolulu,3.0,3,NaN,43,2,0.0007,Mixed Freight,mixed freight,Imports,DMI,2017
4737,Imports from Europe,Honolulu,7.0,7,NaN,43,2,1.0667,Mixed Freight,mixed freight,Imports,DMI,2017
4738,Imports from SW & Central Asia,Honolulu,7.0,7,NaN,43,2,0.0006,Mixed Freight,mixed freight,Imports,DMI,2017
4739,Imports from Eastern Asia,Honolulu,7.0,7,NaN,43,2,0.3654,Mixed Freight,mixed freight,Imports,DMI,2017


In [283]:
df2017 = df2017.drop(columns={'Origin','Destination','fr_inmode', 'dms_mode', 'fr_outmode', 'sctg2', 'trade_type'}).rename(columns={'Tons':'kilotons'})
df2022 = df2022.drop(columns={'Origin','Destination','fr_inmode', 'dms_mode', 'fr_outmode', 'sctg2', 'trade_type'}).rename(columns={'Tons':'kilotons'})
df2017[["Source", "Target"]] = df2017[["Source", "Target"]].replace("NAS", "DMI")
df2022[["Source", "Target"]] = df2022[["Source", "Target"]].replace("NAS", "DMI")

imp_exp_data = pd.concat([dfpre2013, df2017, df2022], ignore_index = True)

imp_exp_data = (
    imp_exp_data
    .groupby(
        ["Material", "Source", "Target", "Year", "Commodity Description"],
        as_index=False
    )["kilotons"]
    .sum()
)
imp = imp_exp_data[imp_exp_data['Source'] == 'Imports']
exp = imp_exp_data[imp_exp_data['Target'] == 'Exports']
exp.to_excel("C:/Users/linsmasa/ODYM-master/revised_model/Imports_Exports/total_exp.xlsx", index=False)
imp.to_excel("C:/Users/linsmasa/ODYM-master/revised_model/Imports_Exports/total_imp.xlsx", index=False)

In [285]:
imp_exp_data

,Material,Source,Target,Year,Commodity Description,kilotons
0,chemicals,DMI,Exports,1997,Basic chemicals,1.959304
1,chemicals,DMI,Exports,1997,Chemical prods.,3.077776
2,chemicals,DMI,Exports,1997,Fertilizers,1.939737
3,chemicals,DMI,Exports,1997,Pharmaceuticals,0.905373
4,chemicals,DMI,Exports,2002,Basic chemicals,1.233983
...,...,...,...,...,...,...
493,wood,Imports,DMI,2012,Wood prods.,19.996474
494,wood,Imports,DMI,2017,Logs and Other Wood in the Rough,13.255100
495,wood,Imports,DMI,2017,Wood Products,25.675100
496,wood,Imports,DMI,2022,Logs,0.133731


### Domestic Extraction

#### 1) Agricultural Production  
Due to data scarcity, often caused by competition issues, there is only limited information on agricultural production in Oahu and Hawaii.
To tackle this challenge, background information from Loke & Leung 2013 is used. The authors investigated the food supply of Hawaii and evaluated insights that are used in this work to estimate agricultural production in Oahu. 

#### 1.1) Calculate domestic food consumption  
Data is based on different data sources (see SI1).

In [332]:
#dfc
input = {
    'Year': [1997, 2002, 2007, 2012, 2017, 2022],
    'kilotons': [493.91, 547.54, 556.60, 599.10, 642.12, 630.24]
}
dfc = pd.DataFrame(input)

In [334]:
dfc

,Year,kilotons
0,1997,493.91
1,2002,547.54
2,2007,556.60
3,2012,599.10
4,2017,642.12
5,2022,630.24


#### 1.2) Extract information on food imports

We generated two different export datasets: one that uses an export share and one that is generated with Holt's smoothing. In the manuscript, we apply the export data that is generated with the share instead of Holt. However, results are compared in SI-A.

In [342]:
imp.head()

,Material,Source,Target,Year,Commodity Description,kilotons
24,chemicals,Imports,DMI,1997,Basic chemicals,7.489495
25,chemicals,Imports,DMI,1997,Chemical prods.,43.320547
26,chemicals,Imports,DMI,1997,Fertilizers,15.004246
27,chemicals,Imports,DMI,1997,Pharmaceuticals,3.913956
28,chemicals,Imports,DMI,2002,Basic chemicals,5.152962


In [344]:
#food item subset
foodimp = imp[(imp['Material'] == "food")]

In [346]:
print(foodimp['Commodity Description'].unique())

['Alcoholic beverages' 'Animal feed' 'Cereal grains' 'Live animals/fish'
 'Meat/seafood' 'Milled grain prods.' 'Other ag prods.' 'Other foodstuffs'
 'Tobacco prods.'
 'Agricultural Products (excludes Animal Feed, Cereal Grains, and Forage Products) '
 'Alcoholic Beverages and Denatured Alcohol '
 'Animal Feed, Eggs, Honey, and Other Products of Animal Origin '
 'Animals and Fish (live) ' 'Cereal Grains (includes seed) '
 'Meat, Poultry, Fish, Seafood, and Their Preparations '
 'Milled Grain Products and Preparations, and Bakery Products '
 'Other Prepared Foodstuffs, Fats and Oils ' 'Tobacco Products ']


In [348]:
foodimp = foodimp[(foodimp['Commodity Description'] != 'Alcoholic beverages') & (foodimp['Commodity Description'] != 'Alcoholic beverages') & (foodimp['Commodity Description'] != 'Tobacco prods.') & (foodimp['Commodity Description'] != 'Alcoholic Beverages and Denatured Alcohol ') & (foodimp['Commodity Description'] != 'Tobacco Products ')]

In [350]:
#foodimp

In [352]:
foodimp =  foodimp.groupby(['Year','Source', 'Target', 'Material']).agg({f'kilotons': 'sum'}).reset_index()

In [354]:
foodimp

,Year,Source,Target,Material,kilotons
0,1997,Imports,DMI,food,576.908950
1,2002,Imports,DMI,food,1041.922126
2,2007,Imports,DMI,food,692.601486
3,2012,Imports,DMI,food,487.933825
4,2017,Imports,DMI,food,529.480500
5,2022,Imports,DMI,food,520.869760


#### 1.3) Extract information on food exports  

In [357]:
foodexp = exp[(exp['Material'] == "food")]
foodexp = foodexp[(foodexp['Commodity Description'] != 'Alcoholic beverages') & (foodexp['Commodity Description'] != 'Alcoholic beverages') & (foodexp['Commodity Description'] != 'Tobacco prods.') & (foodexp['Commodity Description'] != 'Alcoholic Beverages and Denatured Alcohol ') & (foodexp['Commodity Description'] != 'Tobacco Products ')]
foodexp =  foodexp.groupby(['Year','Source', 'Target', 'Material']).agg({f'kilotons': 'sum'}).reset_index()
foodexp

,Year,Source,Target,Material,kilotons
0,1997,DMI,Exports,food,278.079624
1,2002,DMI,Exports,food,504.916831
2,2007,DMI,Exports,food,202.318767
3,2012,DMI,Exports,food,171.447032
4,2017,DMI,Exports,food,225.785100
5,2022,DMI,Exports,food,125.619866


#### 1.4) Extract food waste

This step is added into new_script (MFA model), because total EoL is calculated in the model.

#### 1.5) Calculate Domestic Extraction
DE = DFC + Exports - Imports

In [372]:
display(foodimp, foodexp, dfc)

,Year,Source,Target,Material,kilotons
0,1997,Imports,DMI,food,576.908950
1,2002,Imports,DMI,food,1041.922126
2,2007,Imports,DMI,food,692.601486
3,2012,Imports,DMI,food,487.933825
4,2017,Imports,DMI,food,529.480500
5,2022,Imports,DMI,food,520.869760


,Year,Source,Target,Material,kilotons
0,1997,DMI,Exports,food,278.079624
1,2002,DMI,Exports,food,504.916831
2,2007,DMI,Exports,food,202.318767
3,2012,DMI,Exports,food,171.447032
4,2017,DMI,Exports,food,225.785100
5,2022,DMI,Exports,food,125.619866


,Year,kilotons
0,1997,493.91
1,2002,547.54
2,2007,556.60
3,2012,599.10
4,2017,642.12
5,2022,630.24


In [374]:
de_food = foodexp.copy()
de_food['kilotons'] = dfc['kilotons'] + foodexp['kilotons'] - foodimp['kilotons']
de_food['Source'] = 'DE'
de_food['Target'] = 'DMI'

In [376]:
display(de_food)

,Year,Source,Target,Material,kilotons
0,1997,DE,DMI,food,195.080675
1,2002,DE,DMI,food,10.534704
2,2007,DE,DMI,food,66.317281
3,2012,DE,DMI,food,282.613207
4,2017,DE,DMI,food,338.424600
5,2022,DE,DMI,food,234.990106


#### 2) Mining  
Mining data is obtained on Hawaii level. The share of Oahu's emissions within the mining sector is used as an estimation share for the actual mining of minerals.

USGS data download link:  https://www.usgs.gov/media/files/usgs-aggregates-time-series-data-state-type-and-end-use

In [380]:
share = 0.74830173
df = pd.read_excel("C:/Users/linsmasa/ODYM-master/revised_model/SI2_revised.xlsx", 'USGS_1971-2022')
display(df)

,Table Number,Year,State,Region,Division,Classification,Commodity,Data Description,Designation/Category,Quantity,Total Value,Unit Value,Sort Index
0,1,2022,Alabama,South,East South Central,"Aggregates, construction","Stone, crushed",State totals,Total or average,43800000,688000000,15.72,1
1,1,2022,Alabama,South,East South Central,"Aggregates, construction","Sand and gravel, construction",State totals,Total or average,13500000,157000000,11.64,2
2,2,2022,Alabama,South,East South Central,"Aggregates, construction","Stone, crushed",Type of stone,Limestone,38600000,600000000,15.53,3
3,2,2022,Alabama,South,East South Central,"Aggregates, construction","Stone, crushed",Type of stone,Dolomite,W (Limestone),W (Limestone),W (Limestone),4
4,2,2022,Alabama,South,East South Central,"Aggregates, construction","Stone, crushed",Type of stone,Granite,3420000,57300000,16.76,8
...,...,...,...,...,...,...,...,...,...,...,...,...,...
18185,1,1971,West Virginia,South,South Atlantic,"Aggregates, construction","Sand and gravel, construction",State totals,Total or average,6450000,16800000,NaN,2
18186,1,1971,Wisconsin,Midwest,East North Central,"Aggregates, construction","Stone, crushed",State totals,Total or average,14100000,25100000,NaN,1
18187,1,1971,Wisconsin,Midwest,East North Central,"Aggregates, construction","Sand and gravel, construction",State totals,Total or average,35000000,32700000,NaN,2
18188,1,1971,Wyoming,West,Mountain,"Aggregates, construction","Stone, crushed",State totals,Total or average,2630000,4790000,NaN,1


In [382]:
HI = df[df['State']=='Hawaii']
# based on given data description, Table 1 data needs to be selected, as it shows the total amount of produced minerals.
HI = HI[HI['Table Number']==1]
HI.loc[:, 'Quantity'] = pd.to_numeric(HI['Quantity'], errors='coerce')
HI = HI.groupby('Year').agg({'Quantity': 'sum'}).reset_index()


In [384]:
Oahu = HI.copy()
Oahu['Quantity']=Oahu['Quantity']*share
Oahu['Quantity'] = Oahu['Quantity']/1000  # kilotons
Oahu = Oahu.rename(columns={'Quantity': 'kilotons'})
Oahu['Source'] = "DE"
Oahu['Target'] = "DMI"
Oahu['Material'] = "construction material and minerals"
Oahu.head()

,Year,kilotons,Source,Target,Material
0,1971,4675.389209,DE,DMI,construction material and minerals
1,1972,3810.352409,DE,DMI,construction material and minerals
2,1973,5382.534344,DE,DMI,construction material and minerals
3,1974,5857.705942,DE,DMI,construction material and minerals
4,1975,5589.065621,DE,DMI,construction material and minerals


In [386]:
DE = pd.concat([de_food, Oahu], ignore_index = True)
DE.to_excel("DE_total.xlsx", index=False)

### Preparation of Recycling data

Download link for recycling data: https://www.honolulu.gov/env/ref/res-rates-and-data/
Data used here is based on a pdf, with data transferred to Excel.

In [231]:
#read recycling data
path = r"C:/Users/linsmasa/ODYM-master/new_model/2021_and_previous_years_recycling_data_breakdown.xlsx"
r = pd.read_excel(path)
years = (1997,2002,2007,2012,2017,2022)
# Read the first sheet to start the DataFrame
final_data = pd.read_excel(path, sheet_name=f"{years[0]}")
final_data = final_data.rename(columns={'Material Type': 'Material', 'category': 'Category', 'amount in tons': f'Tons_{years[0]}'})

# Loop through the remaining years to pull 'amount in tons' columns from each sheet
for year in years[1:]:
    gw = pd.read_excel(path, sheet_name=f"{year}")
    gw = gw.rename(columns={'Material Type': 'Material', 'category': 'Category','amount in tons': f'Tons_{year}'})
    
    # Merge on Material and Category columns to keep the structure consistent
    final_data = pd.merge(final_data, gw[['Material', 'Category', f'Tons_{year}']], on=['Material', 'Category'], how='outer')

# Display the final DataFrame
display(final_data)

,Material,Category,Tons_1997,Tons_2002,Tons_2007,Tons_2012,Tons_2017,Tons_2022
0,Paper,Corrugated Cardboard,31400.0,38030.0,48595.0,45235.0,43119.0,37731.0
1,Paper,Newspaper,13200.0,13469.0,14792.0,14415.0,12501.0,6348.0
2,Paper,Office Paper,4500.0,3340.0,3880.0,12548.0,9337.0,5899.0
3,Paper,Other Paper,8400.0,3628.0,12208.0,1326.0,863.0,682.0
4,Metals,ferrous (includes autos),116000.0,106479.0,172215.0,171143.0,111495.0,162366.0
5,Metals,non-ferrous (includes aluminum),9500.0,9272.0,12543.0,14016.0,22409.0,13454.0
6,Glass,Glass,11000.0,11381.0,20745.0,22934.0,14713.0,8939.0
7,Plastic,Plastic,194.0,375.0,4151.0,5903.0,5889.0,3862.0
8,Tires,Tires,3600.0,8027.0,8427.0,12926.0,6477.0,7532.0
9,Auto Batteries,Auto Batteries,3350.0,4167.0,7843.0,2654.0,6859.0,6053.0


Add mapping information for Recycling data.

In [234]:
#sctg2_mapping = "ODYM-master/new_model/FAF_mapping.xlsx" path to sctg2 mapping
rec_mat = pd.read_excel("C:/Users/linsmasa/ODYM-master/new_model/SI2_revised.xlsx", sheet_name = "Recycling")
print(rec_mat.head())

  Material Type                  category Material
0         Paper      Corrugated Cardboard    paper
1         Paper                 Newspaper    paper
2         Paper              Office Paper    paper
3         Paper               Other Paper    paper
4        Metals  ferrous (includes autos)   metals


In [236]:
# Dictionary for description 
mat_map = rec_mat.set_index("category")["Material"]

In [238]:
df = final_data.copy()
df["Material"] = (
    df["Category"]
    .map(mat_map)
    .fillna(df["Material"])
)
display(df)

,Material,Category,Tons_1997,Tons_2002,Tons_2007,Tons_2012,Tons_2017,Tons_2022
0,paper,Corrugated Cardboard,31400.0,38030.0,48595.0,45235.0,43119.0,37731.0
1,paper,Newspaper,13200.0,13469.0,14792.0,14415.0,12501.0,6348.0
2,paper,Office Paper,4500.0,3340.0,3880.0,12548.0,9337.0,5899.0
3,paper,Other Paper,8400.0,3628.0,12208.0,1326.0,863.0,682.0
4,metals,ferrous (includes autos),116000.0,106479.0,172215.0,171143.0,111495.0,162366.0
5,metals,non-ferrous (includes aluminum),9500.0,9272.0,12543.0,14016.0,22409.0,13454.0
6,glass,Glass,11000.0,11381.0,20745.0,22934.0,14713.0,8939.0
7,plastic,Plastic,194.0,375.0,4151.0,5903.0,5889.0,3862.0
8,motorized vehicles,Tires,3600.0,8027.0,8427.0,12926.0,6477.0,7532.0
9,motorized vehicles,Auto Batteries,3350.0,4167.0,7843.0,2654.0,6859.0,6053.0


In [240]:
year_cols = [c for c in df.columns if c.startswith("Tons_")]

# Melt while keeping all other columns
df_long = df.melt(
    id_vars=[c for c in df.columns if c not in year_cols],
    value_vars=year_cols,
    var_name='Year',
    value_name='Tons'
)

# Extract numeric year from 'kg_2018' → 2018
df_long['Year'] = df_long['Year'].str.extract(r'(\d{4})').astype(int)
display(df_long)

,Material,Category,Year,Tons
0,paper,Corrugated Cardboard,1997,31400.0
1,paper,Newspaper,1997,13200.0
2,paper,Office Paper,1997,4500.0
3,paper,Other Paper,1997,8400.0
4,metals,ferrous (includes autos),1997,116000.0
...,...,...,...,...
103,final products,"other reuse (Goodwill, Salvation Army)",2022,37050.0
104,electronics,Electronic Scrap,2022,2291.0
105,wood,Wood waste / pallets,2022,4525.0
106,sewage sludge,sewage sludge,2022,NaN


In [242]:
df_long.to_excel("C:/Users/linsmasa/ODYM-master/new_model/total_recycling.xlsx", index=False)